In [ ]:
!pip install -q sentence-transformers

In [ ]:


from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np

# ---- EDIT THESE ----
DATA_PATH = '/content/drive/MyDrive/nutrilens/only_english.parquet'  # or .csv
TEXT_COL  = 'ingredients_text'
LABEL_COL = 'nova_group'
ID_COL    = 'code'            # barcode / product id
OUT_DIR   = '/content/drive/MyDrive/nutrilens/'
TARGET_N  = 60_000
MIN_PER_CLASS = 5_000         # keeps rare NOVA groups well represented
# --------------------

df = pd.read_parquet(DATA_PATH) if DATA_PATH.endswith('.parquet') else pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).drop_duplicates(subset=[TEXT_COL])
print(df[LABEL_COL].value_counts())

# Stratified sample: proportional share per class, with a floor for rare classes
frac = TARGET_N / len(df)
parts = []
for g, sub in df.groupby(LABEL_COL):
    n = min(len(sub), max(int(len(sub) * frac), MIN_PER_CLASS))
    parts.append(sub.sample(n=n, random_state=42))
sample = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(len(sample)); print(sample[LABEL_COL].value_counts())
sample.to_parquet(OUT_DIR + 'sample.parquet', index=False)

Mounted at /content/drive
nova_group
4    170293
3     30445
1      6795
2       681
Name: count, dtype: int64
63526
nova_group
4    49072
3     8773
1     5000
2      681
Name: count, dtype: int64


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'NeuML/pubmedbert-base-embeddings'
model = SentenceTransformer(MODEL_NAME, device='cuda')
model.half()  # fp16 on GPU: faster, negligible quality change

texts = sample[TEXT_COL].astype(str).tolist()

# encode() batches and sorts by length internally; normalize for cosine/FAISS
emb = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype(np.float32)

print(emb.shape)  # (≈60000, 768)
np.save(OUT_DIR + 'emb_pubmed.npy', emb)
sample[[ID_COL, TEXT_COL, LABEL_COL]].to_parquet(OUT_DIR + 'emb_index.parquet', index=False)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/497 [00:00<?, ?it/s]

(63526, 768)


In [ ]:
!pip install -q xgboost

import numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

OUT_DIR = '/content/drive/MyDrive/nutrilens/'
emb  = np.load(OUT_DIR + 'emb_pubmed.npy')
meta = pd.read_parquet(OUT_DIR + 'emb_index.parquet')
y = meta['nova_group'].astype(int).values - 1          # NOVA 1-4 -> 0-3
idx = np.arange(len(y))

# 70 / 15 / 15, stratified by NOVA group
tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(tmp_idx, test_size=0.50, stratify=y[tmp_idx], random_state=42)
print('train', len(tr_idx), 'val', len(val_idx), 'test', len(test_idx))

clf = xgb.XGBClassifier(
    objective='multi:softprob', num_class=4,
    n_estimators=1000, learning_rate=0.1, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cuda',
    eval_metric='mlogloss', early_stopping_rounds=50,
)
clf.fit(emb[tr_idx], y[tr_idx],
        sample_weight=compute_sample_weight('balanced', y[tr_idx]),
        eval_set=[(emb[val_idx], y[val_idx])], verbose=100)

yt, pred = y[test_idx], clf.predict(emb[test_idx])
print('best iteration:', clf.best_iteration)
print('accuracy   :', round(accuracy_score(yt, pred), 4))
print('macro F1   :', round(f1_score(yt, pred, average='macro'), 4))
print('weighted F1:', round(f1_score(yt, pred, average='weighted'), 4))
print(classification_report(yt, pred, target_names=['NOVA1','NOVA2','NOVA3','NOVA4'], digits=3))
print(confusion_matrix(yt, pred))
clf.save_model(OUT_DIR + 'xgb_pubmed.json')

train 44468 val 9529 test 9529
[0]	validation_0-mlogloss:1.28221
[100]	validation_0-mlogloss:0.26402
[200]	validation_0-mlogloss:0.21766
[300]	validation_0-mlogloss:0.20536
[400]	validation_0-mlogloss:0.20418
[419]	validation_0-mlogloss:0.20445


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [04:08:21] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


best iteration: 369
accuracy   : 0.9293
macro F1   : 0.8215
weighted F1: 0.9297
              precision    recall  f1-score   support

       NOVA1      0.848     0.892     0.869       750
       NOVA2      0.836     0.549     0.663       102
       NOVA3      0.764     0.815     0.789      1316
       NOVA4      0.971     0.959     0.965      7361

    accuracy                          0.929      9529
   macro avg      0.855     0.804     0.821      9529
weighted avg      0.931     0.929     0.930      9529

[[ 669    4   51   26]
 [  13   56   27    6]
 [  59    4 1073  180]
 [  48    3  253 7057]]


In [ ]:
!pip install -q huggingface_hub


In [ ]:
# !pip install -q huggingface_hub

import numpy as np, pandas as pd
from google.colab import userdata
from huggingface_hub import InferenceClient

OUT_DIR = '/content/drive/MyDrive/nutrilens/'
MODEL = 'NeuML/pubmedbert-base-embeddings'
client = InferenceClient(token=userdata.get('HF_TOKEN'))

meta = pd.read_parquet(OUT_DIR + 'emb_index.parquet')
emb  = np.load(OUT_DIR + 'emb_pubmed.npy')
sample_ids = [0, 1, 2, 3, 4]
texts = meta['ingredients_text'].astype(str).iloc[sample_ids].tolist()

def api_embed(text):
    out = np.array(client.feature_extraction(text, model=MODEL), dtype=np.float32)
    print('raw API output shape:', out.shape)
    if out.ndim == 3: out = out[0]              # (1, tokens, 768) -> (tokens, 768)
    if out.ndim == 2 and out.shape[0] > 1:      # token-level: mean-pool to one vector
        out = out.mean(axis=0)
    out = out.reshape(-1)
    return out / np.linalg.norm(out)            # normalize like the local run

for i, t in zip(sample_ids, texts):
    v_api = api_embed(t)
    cos = float(v_api @ emb[i])
    print(f'row {i}: dim={v_api.shape[0]}  cosine(API, local)={cos:.5f}')

raw API output shape: (768,)
row 0: dim=768  cosine(API, local)=0.99996
raw API output shape: (768,)
row 1: dim=768  cosine(API, local)=0.99993
raw API output shape: (768,)
row 2: dim=768  cosine(API, local)=1.00020
raw API output shape: (768,)
row 3: dim=768  cosine(API, local)=0.99982
raw API output shape: (768,)
row 4: dim=768  cosine(API, local)=0.99990


In [ ]:
import os, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

OUT_DIR   = '/content/drive/MyDrive/nutrilens/'
DATA_PATH = '/content/drive/MyDrive/nutrilens/only_english.parquet'   # your full cleaned file
TEXT_COL, LABEL_COL, ID_COL = 'ingredients_text', 'nova_group', 'code'
CHUNK = 50_000
CHUNK_DIR = OUT_DIR + 'full_chunks/'
os.makedirs(CHUNK_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH) if DATA_PATH.endswith('.parquet') else pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
df[[ID_COL, TEXT_COL, LABEL_COL]].to_parquet(OUT_DIR + 'full_index.parquet', index=False)
print('rows:', len(df)); print(df[LABEL_COL].value_counts())

model = SentenceTransformer('NeuML/pubmedbert-base-embeddings', device='cuda').half()
texts = df[TEXT_COL].astype(str).tolist()

for start in range(0, len(texts), CHUNK):
    path = f'{CHUNK_DIR}emb_{start:07d}.npy'
    if os.path.exists(path):
        print('skip', path); continue
    e = model.encode(texts[start:start+CHUNK], batch_size=128, show_progress_bar=True,
                     convert_to_numpy=True).astype(np.float32)
    e /= np.linalg.norm(e, axis=1, keepdims=True)          # normalize in float32
    np.save(path, e); print('saved', path, e.shape)

# stitch chunks into one file
files = sorted(f for f in os.listdir(CHUNK_DIR) if f.endswith('.npy'))
emb_full = np.concatenate([np.load(CHUNK_DIR + f) for f in files])
assert len(emb_full) == len(df)
np.save(OUT_DIR + 'emb_full.npy', emb_full)
print('done', emb_full.shape)

rows: 208214
nova_group
4    170293
3     30445
1      6795
2       681
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

saved /content/drive/MyDrive/nutrilens/full_chunks/emb_0000000.npy (50000, 768)


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

saved /content/drive/MyDrive/nutrilens/full_chunks/emb_0050000.npy (50000, 768)


... (2 outputs left out)

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

saved /content/drive/MyDrive/nutrilens/full_chunks/emb_0150000.npy (50000, 768)


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

saved /content/drive/MyDrive/nutrilens/full_chunks/emb_0200000.npy (8214, 768)
done (208214, 768)


In [ ]:
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

OUT_DIR = '/content/drive/MyDrive/nutrilens/'
emb  = np.load(OUT_DIR + 'emb_full.npy')
meta = pd.read_parquet(OUT_DIR + 'full_index.parquet')
y = meta['nova_group'].astype(int).values - 1
idx = np.arange(len(y))

tr_idx, tmp_idx = train_test_split(idx, test_size=0.30, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(tmp_idx, test_size=0.50, stratify=y[tmp_idx], random_state=42)
np.savez(OUT_DIR + 'split_full.npz', tr=tr_idx, val=val_idx, test=test_idx)
print('train', len(tr_idx), 'val', len(val_idx), 'test', len(test_idx))

clf = xgb.XGBClassifier(
    objective='multi:softprob', num_class=4,
    n_estimators=2000, learning_rate=0.1, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cuda',
    eval_metric='mlogloss', early_stopping_rounds=50,
)
clf.fit(emb[tr_idx], y[tr_idx],
        sample_weight=compute_sample_weight('balanced', y[tr_idx]),
        eval_set=[(emb[val_idx], y[val_idx])], verbose=100)

clf.set_params(device='cpu')                     # silences the device warning
yt, pred = y[test_idx], clf.predict(emb[test_idx])
print('best iteration:', clf.best_iteration)
print('accuracy   :', round(accuracy_score(yt, pred), 4))
print('macro F1   :', round(f1_score(yt, pred, average='macro'), 4))
print('weighted F1:', round(f1_score(yt, pred, average='weighted'), 4))
print(classification_report(yt, pred, target_names=['NOVA1','NOVA2','NOVA3','NOVA4'], digits=3))
print(confusion_matrix(yt, pred))
clf.save_model(OUT_DIR + 'xgb_full.json')

train 145749 val 31232 test 31233
[0]	validation_0-mlogloss:1.28116
[100]	validation_0-mlogloss:0.26172
[200]	validation_0-mlogloss:0.20293
[300]	validation_0-mlogloss:0.17844
[400]	validation_0-mlogloss:0.16477
[500]	validation_0-mlogloss:0.15670
[600]	validation_0-mlogloss:0.15195
[700]	validation_0-mlogloss:0.14890
[800]	validation_0-mlogloss:0.14748
[900]	validation_0-mlogloss:0.14687
[991]	validation_0-mlogloss:0.14695
best iteration: 941
accuracy   : 0.9513
macro F1   : 0.7754
weighted F1: 0.9514
              precision    recall  f1-score   support

       NOVA1      0.835     0.854     0.844      1019
       NOVA2      0.596     0.333     0.428       102
       NOVA3      0.832     0.878     0.855      4567
       NOVA4      0.980     0.971     0.975     25545

    accuracy                          0.951     31233
   macro avg      0.811     0.759     0.775     31233
weighted avg      0.952     0.951     0.951     31233

[[  870     5    98    46]
 [   12    34    45    11]
 [ 

In [ ]:
import re, random, numpy as np, pandas as pd, xgboost as xgb
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score

OUT_DIR = '/content/drive/MyDrive/nutrilens/'
meta  = pd.read_parquet(OUT_DIR + 'full_index.parquet')
split = np.load(OUT_DIR + 'split_full.npz')
test_idx = split['test']
y = meta['nova_group'].astype(int).values - 1

clf = xgb.XGBClassifier(); clf.load_model(OUT_DIR + 'xgb_full.json'); clf.set_params(device='cpu')
model = SentenceTransformer('NeuML/pubmedbert-base-embeddings', device='cuda').half()

def embed(texts):
    e = model.encode(texts, batch_size=128, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    return e / np.linalg.norm(e, axis=1, keepdims=True)

def score(texts, labels, base_pred=None):
    p = clf.predict(embed(texts))
    out = {'acc': accuracy_score(labels, p), 'macroF1': f1_score(labels, p, average='macro')}
    if base_pred is not None: out['agree_w_orig'] = (p == base_pred).mean()
    return out, p

# ---------- Test 1: random character masking ----------
rng = np.random.default_rng(42)
sub = rng.choice(test_idx, size=3000, replace=False)
texts, labels = meta['ingredients_text'].astype(str).values[sub].tolist(), y[sub]
base, base_pred = score(texts, labels)
print('clean:', {k: round(v, 4) for k, v in base.items()})

def mask(t, p, seed):
    r = random.Random(seed); chars = list(t)
    for i, c in enumerate(chars):
        if not c.isspace() and r.random() < p: chars[i] = '#'
    return ''.join(chars)

for p in [0.1, 0.2, 0.3]:
    res, _ = score([mask(t, p, i) for i, t in enumerate(texts)], labels, base_pred)
    print(f'mask {int(p*100)}%:', {k: round(v, 4) for k, v in res.items()})

# ---------- Test 2: additive alias substitution (E-number <-> name) ----------
ALIASES = {
 'E330':'citric acid','E322':'lecithin','E471':'mono- and diglycerides of fatty acids',
 'E415':'xanthan gum','E412':'guar gum','E440':'pectin','E300':'ascorbic acid',
 'E211':'sodium benzoate','E202':'potassium sorbate','E200':'sorbic acid','E250':'sodium nitrite',
 'E621':'monosodium glutamate','E627':'disodium guanylate','E631':'disodium inosinate',
 'E951':'aspartame','E950':'acesulfame potassium','E955':'sucralose','E420':'sorbitol',
 'E407':'carrageenan','E410':'locust bean gum','E414':'gum arabic','E418':'gellan gum',
 'E401':'sodium alginate','E466':'carboxymethyl cellulose','E433':'polysorbate 80',
 'E476':'polyglycerol polyricinoleate','E129':'allura red','E102':'tartrazine',
 'E150a':'caramel color','E160a':'beta-carotene','E270':'lactic acid','E296':'malic acid',
 'E260':'acetic acid','E331':'sodium citrate','E450':'diphosphates','E500':'sodium bicarbonate',
 'E503':'ammonium bicarbonate','E551':'silicon dioxide','E575':'glucono delta-lactone',
 'E320':'bha','E321':'bht',
}
pairs = []
for code, name in ALIASES.items():
    num = code[1:]
    pairs.append((re.compile(rf'\be[\s\-]?{num}\b', re.I), name))   # E-number -> name
    pairs.append((re.compile(rf'\b{re.escape(name)}\b', re.I), code))  # name -> E-number

def swap(t):
    new = t
    for pat, rep in pairs: new = pat.sub(rep, new)
    return new

all_texts = meta['ingredients_text'].astype(str).values[test_idx]
swapped   = [swap(t) for t in all_texts]
changed   = [i for i, (a, b) in enumerate(zip(all_texts, swapped)) if a != b]
changed   = changed[:5000]                                   # cap for speed
print('test products with at least one swap:', len(changed))

o_texts = [all_texts[i] for i in changed]; s_texts = [swapped[i] for i in changed]
o_lab   = y[test_idx][changed]
orig, orig_pred = score(o_texts, o_lab)
alias, _        = score(s_texts, o_lab, orig_pred)
print('alias subset, original :', {k: round(v, 4) for k, v in orig.items()})
print('alias subset, swapped  :', {k: round(v, 4) for k, v in alias.items()})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

clean: {'acc': 0.9547, 'macroF1': 0.6647}
mask 10%: {'acc': 0.8817, 'macroF1': 0.5154, 'agree_w_orig': np.float64(0.894)}
mask 20%: {'acc': 0.834, 'macroF1': 0.4063, 'agree_w_orig': np.float64(0.842)}
mask 30%: {'acc': 0.7883, 'macroF1': 0.3318, 'agree_w_orig': np.float64(0.794)}
test products with at least one swap: 5000
alias subset, original : {'acc': 0.9804, 'macroF1': 0.729}
alias subset, swapped  : {'acc': 0.9296, 'macroF1': 0.5975, 'agree_w_orig': np.float64(0.9436)}


In [ ]:
!pip -q install faiss-cpu
import faiss, numpy as np, pandas as pd

In [ ]:
cols = pd.read_parquet(OUT + 'only_english.parquet').columns.tolist()
print(cols)                                    # check which of product_name / brands / categories_tags exist

names = pd.read_parquet(OUT + 'only_english.parquet',
                        columns=[c for c in ['code','product_name','brands','categories_tags'] if c in cols])
names['code'] = names['code'].astype(str)
meta['code']  = meta['code'].astype(str)
meta = meta.merge(names.drop_duplicates('code'), on='code', how='left')
assert len(meta) == len(emb)                   # row order must still match the embeddings

['code', 'product_name', 'categories_tags', 'ingredients_text', 'ingredient_count', 'additives_n', 'additive_density', 'nutriscore_score', 'monounsaturated-fat_100g', 'sugars_100g', 'polyunsaturated-fat_100g', 'sugar_fiber_ratio', 'carbohydrates_100g', 'fiber_100g', 'saturated-fat_100g', 'sodium_per_calorie', 'allergens_en', 'nova_group']


In [ ]:
# !pip -q install faiss-cpu
# import faiss, numpy as np, pandas as pd
OUT = '/content/drive/MyDrive/nutrilens/'
emb  = np.load(OUT + 'emb_full.npy').astype('float32')
meta = pd.read_parquet(OUT + 'full_index.parquet').reset_index(drop=True)

# bring in product names from the cleaned parquet (left merge keeps row order)
names = pd.read_parquet(OUT + 'only_english.parquet', columns=['code', 'product_name'])
names['code'] = names['code'].astype(str)
meta['code']  = meta['code'].astype(str)
meta = meta.merge(names.drop_duplicates('code'), on='code', how='left')
assert len(meta) == len(emb)
nova = meta['nova_group'].astype(int).values

indexes, ids = {}, {}
for g in [1, 2, 3, 4]:
    idx = np.where(nova == g)[0]
    ix = faiss.IndexHNSWFlat(emb.shape[1], 32, faiss.METRIC_INNER_PRODUCT)  # normalized vectors, so inner product = cosine
    ix.hnsw.efConstruction = 100
    ix.add(emb[idx])
    indexes[g], ids[g] = ix, idx

def recommend(vec, query_group, k=5, thresh=0.8):
    vec = vec.reshape(1, -1).astype('float32')
    cands = []
    for g in range(1, query_group):                      # only lower NOVA groups
        sims, nn = indexes[g].search(vec, 20)
        for s, j in zip(sims[0], nn[0]):
            if s >= thresh: cands.append((s, ids[g][j]))
    cands.sort(reverse=True)
    out, seen = [], set()
    for s, r in cands:
        txt = meta.loc[r, 'ingredients_text']
        if txt in seen: continue                         # skip duplicate ingredient lists
        seen.add(txt)
        out.append((round(float(s), 3), int(nova[r]), meta.loc[r, 'product_name'], txt[:80]))
        if len(out) == k: break
    return out

# demo on a few NOVA 4 products
for i in np.random.default_rng(0).choice(np.where(nova == 4)[0], 3, replace=False):
    print('\nQUERY:', meta.loc[i, 'product_name'], '|', meta.loc[i, 'ingredients_text'][:80])
    for r in recommend(emb[i], 4): print('  ', r)


QUERY: Cauliflower, cheese & bacon veggie tots, cauliflower, cheese & bacon | cauliflow soybean oil modifi food starch water cornstarch cheddar chees milk cul
   (0.912, 3, 'English muffins', 'water enrich flour wheat flour malt barley flour niacin ferrou sulfat thiamin mo')
   (0.886, 3, 'Pizza bagels', 'bagel enrich wheat flour wheat flour malt barley flour niacin reduc iron thiamin')
   (0.882, 3, 'Macabee, mozzarella cheese pizza', 'crust enrich wheat flour wheat flour malt barley flour niacin reduc iron thiamin')
   (0.882, 3, "Reggio's, Turkey Sausage Pizza", 'crust flour wheat flour enrich niacin reduc iron thiamin mononitr riboflavin fol')
   (0.877, 3, 'Multi-Grain English Muffins', 'bleach wheat flour whole wheat flour water grain seed blend sunflow seed corn me')

QUERY: Organic grilled breast strips | organ boneless skinless chicken breast rib meat water contain less organ corn st

QUERY: Organic Kids Vitamin D3 | organ tapioca syrup organ sugar organ lemon juic concentr p

In [ ]:
from collections import Counter

# add categories_tags to meta (row order preserved)
if 'categories_tags' not in meta.columns:
    cat = pd.read_parquet(OUT + 'only_english.parquet', columns=['code', 'categories_tags'])
    cat['code'] = cat['code'].astype(str); meta['code'] = meta['code'].astype(str)
    meta = meta.merge(cat.drop_duplicates('code'), on='code', how='left')
assert len(meta) == len(emb)

def to_set(v):
    if v is None or isinstance(v, float): return frozenset()
    if isinstance(v, str): return frozenset(t.strip() for t in v.split(',') if t.strip())
    return frozenset(v)

cats = [to_set(v) for v in meta['categories_tags']]
tag_freq = Counter(t for s in cats for t in s)
for g in indexes: indexes[g].hnsw.efSearch = 256          # needed for k=200 searches

def top_categories(vec, exclude=None, n=20):
    hits = []
    for g in range(1, 5):                                  # neighbors from the whole catalog, any NOVA group
        sims, nn = indexes[g].search(vec, 10)
        hits += [(s, ids[g][j]) for s, j in zip(sims[0], nn[0])]
    hits.sort(reverse=True)
    hits = [r for s, r in hits if r != exclude][:n]
    votes = Counter(t for r in hits for t in cats[r])
    ok = [t for t, c in votes.items() if c >= n / 2]       # majority of neighbors share it
    ok.sort(key=lambda t: tag_freq[t])                     # most specific first
    return ok[:3]

def recommend(vec, query_group, exclude=None, k=5):
    vec = vec.reshape(1, -1).astype('float32')
    tags = top_categories(vec, exclude)
    cands = []
    for g in range(1, query_group):                        # lower NOVA groups only
        sims, nn = indexes[g].search(vec, 200)
        cands += [(s, ids[g][j]) for s, j in zip(sims[0], nn[0])]
    cands.sort(reverse=True)
    for tag in tags:                                       # most specific category, then broader
        out, seen = [], set()
        for s, r in cands:
            if tag not in cats[r]: continue
            txt = meta.loc[r, 'ingredients_text']
            if txt in seen: continue
            seen.add(txt)
            out.append((round(float(s), 3), int(nova[r]), meta.loc[r, 'product_name']))
            if len(out) == k: break
        if out: return tag, out
    return (tags[0] if tags else None), []

for i in np.random.default_rng(0).choice(np.where(nova == 4)[0], 3, replace=False):
    tag, recs = recommend(emb[i], nova[i], exclude=i)
    print('\nQUERY:', meta.loc[i, 'product_name'], '| category:', tag)
    for r in recs: print('  ', r)


QUERY: Cauliflower, cheese & bacon veggie tots, cauliflower, cheese & bacon | category: None

QUERY: Organic grilled breast strips | category: None

QUERY: Organic Kids Vitamin D3 | category: None


In [ ]:
from collections import Counter, defaultdict

# categories_tags merged in earlier; make sure it's there
if 'categories_tags' not in meta.columns:
    cat = pd.read_parquet(OUT + 'only_english.parquet', columns=['code', 'categories_tags'])
    cat['code'] = cat['code'].astype(str); meta['code'] = meta['code'].astype(str)
    meta = meta.merge(cat.drop_duplicates('code'), on='code', how='left')
assert len(meta) == len(emb)

def to_set(v):
    if v is None or isinstance(v, float): return frozenset()
    if isinstance(v, str): return frozenset(t.strip() for t in v.split(',') if t.strip())
    return frozenset(v)

cats = [to_set(v) for v in meta['categories_tags']]
tag_freq = Counter(t for s in cats for t in s)
tag_rows = defaultdict(list)                                  # tag -> list of row numbers
for r, s in enumerate(cats):
    for t in s: tag_rows[t].append(r)
ingredients = meta['ingredients_text'].astype(str).values
print('products with categories:', sum(len(s) > 0 for s in cats), 'of', len(cats))

def recommend(vec, tags, query_group, k=5):
    tags = sorted(tags, key=lambda t: tag_freq[t])            # most specific tag first
    results = []
    for g in range(query_group - 1, 0, -1):                   # e.g. NOVA 4 -> 3, 2, 1
        best = []
        for t in tags:
            rows = np.array(tag_rows[t])
            rows = rows[nova[rows] == g]                      # only this NOVA group
            if len(rows) == 0: continue
            sims = emb[rows] @ vec                            # cosine similarity
            out, seen = [], set()
            for o in np.argsort(-sims):
                r = rows[o]
                if ingredients[r] in seen: continue           # skip duplicate ingredient lists
                seen.add(ingredients[r])
                out.append({'product': meta.loc[r, 'product_name'], 'nova': g,
                            'similarity': round(float(sims[o]), 3),
                            'match_pct': round(float(sims[o]) * 100, 1),
                            'category': t})
                if len(out) == k: break
            if len(out) >= k: best = out; break               # specific enough and enough results
            if len(out) > len(best): best = out
        results += best
    return results

valid = [i for i in np.where(nova == 4)[0] if len(cats[i]) > 0]
for i in np.random.default_rng(0).choice(valid, 4, replace=False):
    print('\nQUERY:', meta.loc[i, 'product_name'], meta.loc[i, 'nova_group'])
    for r in recommend(emb[i], cats[i], nova[i]): print('  ', r)

products with categories: 180960 of 208214

QUERY: Vegetable broth 4
   {'product': 'Organic Vegetable Broth', 'nova': 3, 'similarity': 0.835, 'match_pct': 83.5, 'category': 'en:vegetable-broths'}
   {'product': 'Vegetable Broth', 'nova': 3, 'similarity': 0.821, 'match_pct': 82.1, 'category': 'en:vegetable-broths'}
   {'product': 'Organic Vegetable Broth Low Sodium', 'nova': 3, 'similarity': 0.819, 'match_pct': 81.9, 'category': 'en:vegetable-broths'}
   {'product': 'Vegetable Broth', 'nova': 3, 'similarity': 0.819, 'match_pct': 81.9, 'category': 'en:vegetable-broths'}
   {'product': 'Organic Low Sodium vegetable broth', 'nova': 3, 'similarity': 0.783, 'match_pct': 78.3, 'category': 'en:vegetable-broths'}
   {'product': 'Organic golden flaxseed spread with dates and orange juice', 'nova': 2, 'similarity': 0.764, 'match_pct': 76.4, 'category': 'en:plant-based-foods'}
   {'product': 'Extra Virgin Olive Oil, Orange Rosemary', 'nova': 2, 'similarity': 0.764, 'match_pct': 76.4, 'category': 

In [ ]:
!pip -q install pymupdf4llm langchain-text-splitters rank_bm25


In [ ]:
import glob, os, re, pickle, numpy as np, faiss, pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import fitz, glob, os, pandas as pd


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [ ]:



PDF_DIR = OUT + 'fda_pdfs/'
MIN_CHARS = 100

rows = []
for path in sorted(glob.glob(PDF_DIR + '*.pdf')):
    doc = fitz.open(path)
    for i, page in enumerate(doc):
        text = page.get_text().strip()
        area = page.rect.width * page.rect.height
        img_area = sum(fitz.Rect(im['bbox']).get_area() for im in page.get_image_info())
        try:    n_tables = len(page.find_tables().tables)
        except Exception: n_tables = -1
        rows.append({'source': os.path.basename(path), 'page': i + 1, 'chars': len(text),
                     'image_cover': round(img_area / area, 2), 'tables': n_tables})

audit = pd.DataFrame(rows)
audit['machine_readable'] = audit['chars'] >= MIN_CHARS
audit['scan_suspect'] = (~audit['machine_readable']) & (audit['image_cover'] > 0.5)

summary = audit.groupby('source').agg(
    pages=('page', 'count'),
    pct_machine_readable=('machine_readable', lambda s: round(100 * s.mean(), 1)),
    scan_suspect_pages=('scan_suspect', 'sum'),
    pages_with_tables=('tables', lambda s: int((s > 0).sum())),
    median_chars=('chars', 'median'))
print(summary)
print('\nOverall % machine readable:', round(100 * audit['machine_readable'].mean(), 1))

audit.to_csv(OUT + 'pdf_audit.csv', index=False)
audit[~audit['machine_readable']].head(20)   # look at the low-text pages

                                                   pages  \
source                                                     
21 CFR Part 170 (up to date as of 11-17-2026).pdf     33   
21 CFR Part 172 (up to date as of 11-17-2026).pdf    178   
21 CFR Part 173 (up to date as of 11-17-2026).pdf     52   
21 CFR Part 182 (up to date as of 11-17-2026).pdf     27   
21 CFR Part 74 (up to date as of 11-17-2026).pdf      68   
CFR-2025-title21-vol1-part73.pdf                      58   
Food-Ingredients-and-Colors-(PDF).pdf                  8   

                                                   pct_machine_readable  \
source                                                                    
21 CFR Part 170 (up to date as of 11-17-2026).pdf                 100.0   
21 CFR Part 172 (up to date as of 11-17-2026).pdf                 100.0   
21 CFR Part 173 (up to date as of 11-17-2026).pdf                 100.0   
21 CFR Part 182 (up to date as of 11-17-2026).pdf                 100.0   
21 CFR Pa

,source,page,chars,image_cover,tables,machine_readable,scan_suspect


In [ ]:
import re, pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)   # fallback only, for long sections

SECTION_RE = re.compile(r'^\W*(§+\s*\d+\.\d+)\s*(.*)$')   # CFR section lines like "§ 172.515 Synthetic flavoring..."
HEADING_RE = re.compile(r'^#{1,3}\s+(.+)$')                # markdown headings (brochure etc.)
clean = lambda s: re.sub(r'[*#_`]', '', s).strip()[:150]

chunks = []
for path in sorted(glob.glob(PDF_DIR + '*.pdf')):
    src = os.path.basename(path)
    pages = pymupdf4llm.to_markdown(path, page_chunks=True)   # one entry per page, in page order
    section = '(front matter)'                                # current section carries across pages
    for pno, pg in enumerate(pages, 1):
        segs, buf = [], []
        for line in pg['text'].split('\n'):
            m = SECTION_RE.match(line)
            h = HEADING_RE.match(line)
            if m or h:                                        # new section starts: close the previous segment
                if buf: segs.append((section, '\n'.join(buf)))
                section = clean(f"{m.group(1)} {m.group(2)}") if m else clean(h.group(1))
                buf = [line]
            else:
                buf.append(line)
        if buf: segs.append((section, '\n'.join(buf)))

        for sec, text in segs:
            text = text.strip()
            if len(text) < 30: continue                       # drop empty/noise segments
            parts = [text] if len(text) <= 1000 else splitter.split_text(text)
            for p in parts:
                chunks.append({'chunk_id': len(chunks), 'source': src, 'page': pno,
                               'section': sec, 'text': p, 'n_chars': len(p)})

chunks_df = pd.DataFrame(chunks)
chunks_df.to_pickle(OUT + 'rag_chunks.pkl')

print(len(chunks_df), 'chunks')
print(chunks_df.groupby('source').size())
print(chunks_df['n_chars'].describe())
chunks_df.sample(5, random_state=0)[['source', 'page', 'section', 'text']]


=== Document parser messages ===
Using Tesseract for OCR processing.


/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=11/12.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=11/12.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=5/6.
2137 chunks
source
21 CFR Part 170 (up to date as of 11-17-2026).pdf    214
21 CFR Part 172 (up to date as of 11-17-2026).pdf    789
21 CFR Part 173 (up to date as of 11-17-2026).pdf    283
21 CFR Part 182 (up to date as of 11-17-2026).pdf    152
21 CFR Part 74 (up to date as of 11-17-2026).pdf     287
CFR-2025-title21-vol1-part73.pdf                     373
Food-Ingredients-and-Colors-(PDF).pdf                 39
dtype: int64
c

,source,page,section,text
810,21 CFR Part 172 (up to date as of 11-17-2026).pdf,138,§ 172.842 Sorbitan monostearate.,- (ii) It is used with polysorbate 60 in any c...
1785,CFR-2025-title21-vol1-part73.pdf,9,§ 73.70 Calcium carbonate.,### **§ 73.70 Calcium carbonate.** \n\n(a) _Id...
2108,Food-Ingredients-and-Colors-(PDF).pdf,2,ingredients &colors,**2**
2099,Food-Ingredients-and-Colors-(PDF).pdf,1,ingredients &colors,## **ingredients &colors** \n\nInternational F...
1638,21 CFR Part 74 (up to date as of 11-17-2026).pdf,48,§ 74.2053 D&C Black No. 3.,- (d) Labeling. The label of the color additiv...


In [ ]:
!pip install -q rank_bm25

In [ ]:


import numpy as np, faiss, pandas as pd
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

chunks_df = pd.read_pickle(OUT + 'rag_chunks.pkl')
chunk_texts = chunks_df['text'].tolist()

# same embedding model as product embeddings
model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
chunk_emb = model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True).astype('float32')
np.save(OUT + 'rag_emb.npy', chunk_emb)

# FAISS HNSW
index = faiss.IndexHNSWFlat(chunk_emb.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = 100
index.hnsw.efSearch = 50
index.add(chunk_emb)

# BM25
tokenized = [text.lower().split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized)

print(f"FAISS: {index.ntotal} vectors, dim {chunk_emb.shape[1]}")
print(f"BM25:  {len(tokenized)} documents")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

FAISS: 2137 vectors, dim 768
BM25:  2137 documents


In [ ]:
from huggingface_hub import InferenceClient
from google.colab import userdata

In [ ]:


HF_TOKEN = userdata.get('HF_TOKEN')       # set in Colab Secrets
# client = InferenceClient(model="meta-llama/Meta-Llama-3.1-8B-Instruct", token=HF_TOKEN)
client = InferenceClient(model="meta-llama/Llama-3.1-8B-Instruct", token=HF_TOKEN)


# ── retrieval ────────────────────────────────────────────────────
def retrieve(query, k=5, top_n=20, c=60):
    """Hybrid semantic + BM25 retrieval, fused with Reciprocal Rank Fusion."""
    # semantic arm
    q_emb = model.encode([query], normalize_embeddings=True).astype('float32')
    sem_scores, sem_ids = index.search(q_emb, top_n)
    sem_ids, sem_scores = sem_ids[0], sem_scores[0]
    assert (sem_ids >= 0).all(), "semantic arm returned empty results"

    # BM25 arm
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_top = np.argsort(-bm25_scores)[:top_n]

    # RRF fusion
    rrf = {}
    for rank, idx in enumerate(sem_ids):
        rrf[int(idx)] = rrf.get(int(idx), 0) + 1 / (c + rank + 1)
    for rank, idx in enumerate(bm25_top):
        rrf[int(idx)] = rrf.get(int(idx), 0) + 1 / (c + rank + 1)

    ranked = sorted(rrf.items(), key=lambda x: -x[1])[:k]
    results = []
    for idx, score in ranked:
        row = chunks_df.iloc[idx]
        results.append({'chunk_id': idx, 'text': row['text'],
                        'source': row['source'], 'page': int(row['page']),
                        'section': row['section'], 'rrf_score': round(score, 4)})
    return results


# ── prompt ───────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
# Role
You are a food science expert. Your job is to explain why a food product \
received its NOVA processing classification, based on its ingredients and \
FDA regulatory document excerpts provided below.

# NOVA Classification Definitions
- NOVA 1 (Unprocessed / Minimally Processed): Natural foods altered only by \
drying, crushing, filtering, pasteurizing, freezing, or fermenting. No added \
substances. Examples: fresh fruit, milk, eggs, plain grains, meat, legumes.
- NOVA 2 (Processed Culinary Ingredients): Substances extracted and purified \
from NOVA 1 foods, used in cooking but rarely eaten alone. Examples: oils, \
butter, sugar, honey, salt, flour, starch.
- NOVA 3 (Processed Foods): NOVA 1 foods modified by adding NOVA 2 \
ingredients using simple methods like canning, bottling, or baking. Typically \
2-3 ingredients, still recognizable as the original food. Examples: canned \
vegetables with salt, cheese, cured meat, fresh bread.
- NOVA 4 (Ultra-Processed): Industrial formulations with five or more \
ingredients, typically including additives not used in home cooking — \
emulsifiers, humectants, flavor enhancers, colorants, sweeteners. Designed \
for convenience and hyper-palatability. Examples: soft drinks, packaged \
snacks, instant noodles, reconstituted meat products.

# Rules
1. ONLY use information from the provided FDA document excerpts. Do not use \
outside knowledge about specific ingredients.
2. Cite every claim about an additive as [Source, p.XX, Section].
3. If an ingredient is not covered in the provided excerpts, state that it \
was not found in the provided documents. Do not guess or speculate.
4. Keep the explanation to 3-5 sentences. It will appear on a product card.
5. Start with the NOVA classification and a one-line reason, then explain \
the key additives found, then briefly note unprocessed base ingredients.
6. Do not repeat the ingredients list back. Go straight to the explanation.
7. If the provided excerpts do not contain information about any of the \
product's key ingredients, say: "The provided FDA documents do not contain \
sufficient information about this product's ingredients to provide a \
reliable explanation." Do not force an explanation from irrelevant excerpts.
8. Write one plain paragraph. No bullet points or lists, and never repeat\
a sentence.

# Examples

Ingredients: water, oat flour, xylitol, carrageenan, natural flavor
NOVA Group: 4

Explanation: This product is classified as NOVA 4 (ultra-processed) due to \
the presence of industrial additives not used in home cooking. \
Xylitol is a sugar alcohol permitted as a sweetener in food \
[21 CFR Part 172.pdf, p.45, § 172.395]. Carrageenan is used as a \
stabilizer and thickener [21 CFR Part 172.pdf, p.112, § 172.620]. \
The base ingredients (water, oat flour) are minimally processed, but \
these additives indicate industrial formulation.

Ingredients: fresh whole milk
NOVA Group: 1

Explanation: This product is classified as NOVA 1 (unprocessed or \
minimally processed). Fresh whole milk is a natural food that has only \
undergone pasteurization. No ingredients were found in the provided \
FDA additive documents."""


def explain(ingredients, nova_group, retrieved_chunks):
    """Generate explanation via Llama 3.1 8B Instruct."""
    context = ""
    for i, ch in enumerate(retrieved_chunks, 1):
        context += (f"\n[Excerpt {i} | Source: {ch['source']} "
                    f"| Page: {ch['page']} | Section: {ch['section']}]\n"
                    f"{ch['text']}\n")

    user_msg = f"""FDA Document Excerpts:
{context}

Product Ingredients: {ingredients}
Predicted NOVA Group: {nova_group}

Based only on the excerpts above, explain why this product received this NOVA classification."""

    response = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ],
        max_tokens=250,
        temperature=0.2,
        frequency_penalty=0.6
    )
    text = response.choices[0].message.content.strip()
    if response.choices[0].finish_reason == 'length' and '.' in text:
        text = text[:text.rfind('.') + 1]
    return text


    # response = client.chat_completion(
    #     messages=[
    #         {"role": "system", "content": SYSTEM_PROMPT},
    #         {"role": "user", "content": user_msg}
    #     ],
    #     max_tokens=250,
    #     temperature=0.2
    # )

    # return response.choices[0].message.content


# ── quick test ───────────────────────────────────────────────────
test_ingredients = "water, sugar, citric acid, sodium benzoate, natural flavor, potassium sorbate"
test_nova = 4

retrieved = retrieve(test_ingredients, k=5)
print("Retrieved chunks:")
for ch in retrieved:
    print(f"  [{ch['source']}, p.{ch['page']}, {ch['section']}] (RRF: {ch['rrf_score']})")

print()
explanation = explain(test_ingredients, test_nova, retrieved)
print("Explanation:")
print(explanation)

Retrieved chunks:
  [CFR-2025-title21-vol1-part73.pdf, p.10, § 73.85 Caramel.] (RRF: 0.031)
  [Food-Ingredients-and-Colors-(PDF).pdf, p.3, ingredients &colors] (RRF: 0.0301)
  [Food-Ingredients-and-Colors-(PDF).pdf, p.3, ingredients &colors] (RRF: 0.0274)
  [21 CFR Part 74 (up to date as of 11-17-2026).pdf, p.58, § 74.2705 FD&C Yellow No. 5.] (RRF: 0.0261)
  [21 CFR Part 172 (up to date as of 11-17-2026).pdf, p.92, § 172.585 Sugar beet extract flavor base.] (RRF: 0.0164)

Explanation:
This product is classified as NOVA 4 (ultra-processed) due to the presence of industrial additives not used in home cooking. Sodium benzoate is a preservative permitted in food [Excerpt 2, Page 3, Section: ingredients & colors]. Potassium sorbate is also a preservative [Excerpt 2, Page 3, Section: ingredients & colors]. Natural flavor is a broad term that can include artificial flavor enhancers [Excerpt 3, Page 3, Section: ingredients & colors]. The base ingredients (water, sugar) are minimally processed,

In [ ]:
if 'product_name' not in meta.columns:
    names = pd.read_parquet(OUT + 'only_english.parquet', columns=['code', 'product_name'])
    names['code'] = names['code'].astype(str); meta['code'] = meta['code'].astype(str)
    meta = meta.merge(names.drop_duplicates('code'), on='code', how='left')

In [ ]:
# --- Cell 5: Quick eval batch ---
import random, time

random.seed(42)
sample = meta.sample(50, random_state=42).reset_index(drop=True)

results = []
for i, row in sample.iterrows():
    ing = row['ingredients_text']
    ng = int(row['nova_group'])

    retrieved = retrieve(ing, k=5)
    top_rrf = retrieved[0]['rrf_score'] if retrieved else 0

    # confidence gate — skip LLM if retrieval is too weak
    if top_rrf < 0.005:
        expl = "Insufficient information found in FDA documents to explain this product's classification."
    else:
        try:
            expl = explain(ing, ng, retrieved)
        except Exception as e:
            expl = f"[ERROR] {e}"

    results.append({
        'product': row['product_name'],
        'nova_group': ng,
        'ingredients': ing[:200],
        'top_rrf_score': top_rrf,
        'n_chunks_retrieved': len(retrieved),
        'explanation': expl
    })

    if (i + 1) % 10 == 0:
        print(f"{i + 1}/50 done")
    time.sleep(1)   # don't hammer the API

eval_df = pd.DataFrame(results)
eval_df.to_csv(OUT + 'rag_eval_sample.csv', index=False)

print(f"\nDone. {len(eval_df)} products evaluated.")
print(f"Skipped (low confidence): {(eval_df['explanation'].str.startswith('Insufficient')).sum()}")
print(f"Errors: {(eval_df['explanation'].str.startswith('[ERROR]')).sum()}")
print(f"\nNOVA distribution in sample:")
print(eval_df['nova_group'].value_counts().sort_index())

# show a few
eval_df[['product', 'nova_group', 'top_rrf_score', 'explanation']].head(5)

10/50 done
20/50 done
30/50 done
40/50 done
50/50 done

Done. 50 products evaluated.
Skipped (low confidence): 0
Errors: 0

NOVA distribution in sample:
nova_group
1     3
3     7
4    40
Name: count, dtype: int64


,product,nova_group,top_rrf_score,explanation
0,"Kielbasa,Polska,Gm,5:1,6"""",10#,Z",4,0.0304,This product is classified as NOVA 4 (ultra-pr...
1,Corn Chex,3,0.0325,This product is classified as NOVA 3 (processe...
2,"Whole foods market, marinade, original teriyaki",4,0.0284,This product is classified as NOVA 4 (ultra-pr...
3,Herb Original Crackers,3,0.0289,This product is classified as NOVA 3 (processe...
4,Beef & broccoli rice bowl,4,0.0164,This product is classified as NOVA 4 (ultra-pr...


In [ ]:
eval_df[['product', 'nova_group', 'top_rrf_score', 'explanation']].head(10)

,product,nova_group,top_rrf_score,explanation
0,"Kielbasa,Polska,Gm,5:1,6"""",10#,Z",4,0.0304,This product is classified as NOVA 4 (ultra-pr...
1,Corn Chex,3,0.0325,This product is classified as NOVA 3 (processe...
2,"Whole foods market, marinade, original teriyaki",4,0.0284,This product is classified as NOVA 4 (ultra-pr...
3,Herb Original Crackers,3,0.0289,This product is classified as NOVA 3 (processe...
4,Beef & broccoli rice bowl,4,0.0164,This product is classified as NOVA 4 (ultra-pr...
5,"Simply indulgent gourmet, glazed cookies, cran...",4,0.0320,This product is classified as NOVA 4 (ultra-pr...
6,"Monterey jack cheese with jalapeno peppers, ho...",3,0.0164,This product is classified as NOVA 3 (processe...
7,Oat Bread,4,0.0325,NOVA 4 (ultra-processed) due to the presence o...
8,Vanilla Sandwich Cookies,4,0.0323,This product is classified as NOVA 4 (ultra-pr...
9,Kashi Cookies Oatmeal Raisin 8.5oz,4,0.0318,This product is classified as NOVA 4 (ultra-pr...
